In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import os

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
DATASET_PATH = "../datasets/face_images/ham10000"
IMG_SIZE = 224
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

val_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

print(f"Classes: {train_generator.class_indices}")
print(f"Training samples: {train_generator.samples}")

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(train_generator.num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print(model.summary())

In [ ]:
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)

print("Training complete")
print(f"Final accuracy: {history.history['accuracy'][-1]}")

In [ ]:
os.makedirs("../models_saved", exist_ok=True)

model.save("../models_saved/face_model_keras")

converter = tf.lite.TFLiteConverter.from_saved_model(
    "../models_saved/face_model_keras"
)
tflite_model = converter.convert()

with open("../models_saved/face_model.tflite", "wb") as f:
    f.write(tflite_model)

import json
class_indices = train_generator.class_indices
class_labels = {v: k for k, v in class_indices.items()}
with open("../models_saved/face_class_labels.json", "w") as f:
    json.dump(class_labels, f)

print("TFLite model saved")
print(f"Classes: {class_labels}")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

interpreter = tf.lite.Interpreter(
    model_path="../models_saved/face_model.tflite"
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

test_img_path = "../datasets/face_images/ham10000/mel"
test_img_file = os.listdir(test_img_path)[0]

img = image.load_img(
    os.path.join(test_img_path, test_img_file),
    target_size=(224, 224)
)
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0).astype(np.float32)

interpreter.set_tensor(input_details[0]["index"], img_array)
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]["index"])[0]
predicted_idx = np.argmax(output)
confidence = round(float(output[predicted_idx]) * 100, 2)

print(f"Predicted: {class_labels[predicted_idx]} — {confidence}%")